In [ ]:
# DSC 540
# Project Milestone 4
# Mahad Farah

In [1]:

import pandas as pd
import requests

url = "https://en.wikipedia.org/wiki/List_of_countries_by_carbon_dioxide_emissions"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()

tables = pd.read_html(response.text)
df_raw = tables[0].copy()

print("Original columns (MultiIndex):")
print(df_raw.columns)


Original columns (MultiIndex):
MultiIndex([(                                 'Location',          'Location'),
            (                          '% ofglobaltotal',   '% ofglobaltotal'),
            ('Fossil emissions(1,000,000 tons per year)',              '2023'),
            ('Fossil emissions(1,000,000 tons per year)',              '2000'),
            (                        '% changefrom 2000', '% changefrom 2000')],
           )


In [2]:
# Step #1: Flatten MultiIndex and rename columns
# What it does: Converts the two-row header (MultiIndex) into single string column names and then maps them to clear labels.

flat_cols = [
    "_".join([str(level).strip() for level in col if str(level).strip() != ""])
    for col in df_raw.columns
]

df_raw.columns = flat_cols

print("Flattened columns:")
print(df_raw.columns)

df = df_raw.copy()

# Build a flexible rename map based on keywords
rename_map = {}
for col in df.columns:
    lc = col.lower()
    if "location" in lc:
        rename_map[col] = "Location"
    elif "% ofglobaltotal" in lc or "% of global total" in lc:
        rename_map[col] = "Percent_Global_Total"
    elif "2023" in lc:
        rename_map[col] = "Emissions_2023_Mt"
    elif "2000" in lc and "change" not in lc:
        rename_map[col] = "Emissions_2000_Mt"
    elif "changefrom 2000" in lc or "change from 2000" in lc:
        rename_map[col] = "Percent_Change_from_2000"

df = df.rename(columns=rename_map)

print("Renamed columns:")
print(df.columns)
df.head()


Flattened columns:
Index(['Location_Location', '% ofglobaltotal_% ofglobaltotal',
       'Fossil emissions(1,000,000 tons per year)_2023',
       'Fossil emissions(1,000,000 tons per year)_2000',
       '% changefrom 2000_% changefrom 2000'],
      dtype='object')
Renamed columns:
Index(['Location', 'Percent_Global_Total', 'Emissions_2023_Mt',
       'Emissions_2000_Mt', 'Percent_Change_from_2000'],
      dtype='object')


,Location,Percent_Global_Total,Emissions_2023_Mt,Emissions_2000_Mt,Percent_Change_from_2000
0,World,100%,39023.94,25725.44,+52%
1,China,34.0%,13259.64,3666.95,+262%
2,United States,12.0%,4682.04,5928.97,−21%
3,India,7.6%,2955.18,995.65,+197%
4,European Union,6.4%,2512.07,3563.26,−30%


In [3]:
# Step #2: Clean Location names
# What it does: Removes footnote markers like “[1]” and extra spaces from the Location column to get a clean location name.


df["Location"] = (
    df["Location"]
    .astype(str)
    .str.replace(r"\[[^\]]*\]", "", regex=True)  
    .str.strip()
)

df[["Location"]].head()


,Location
0,World
1,China
2,United States
3,India
4,European Union


In [4]:
# Step #3: Drop empty rows and duplicates
# What it does: Removes rows that are entirely NaN and rows that are exact duplicates so only important records are kept.

before_shape = df.shape
df = df.dropna(how="all")
df = df.drop_duplicates()
after_shape = df.shape

print("Shape before:", before_shape, "Shape after:", after_shape)


Shape before: (212, 5) Shape after: (212, 5)


In [5]:
# Step #4: Convert numeric looking columns to numeric dtype
# What it does: Strips commas, percent symbols, etc., and converts columns to floats so that calculations and visualizations can be completed.

numeric_cols = [
    "Emissions_2023_Mt",
    "Emissions_2000_Mt",
    "Percent_Global_Total",
    "Percent_Change_from_2000",
]

for col in numeric_cols:
    if col not in df.columns:
        continue

    cleaned = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(cleaned, errors="coerce")

df[[c for c in numeric_cols if c in df.columns]].head()
df[[c for c in numeric_cols if c in df.columns]].dtypes


Emissions_2023_Mt           float64
Emissions_2000_Mt           float64
Percent_Global_Total        float64
Percent_Change_from_2000    float64
dtype: object

In [6]:
# Step #5: Remove non-country rows
# What it does: Filters out summary rows like "World" or "European Union" so each remaining row represents an actual country.

aggregate_labels = [
    "World",
    "Global total",
    "European Union",
    "International aviation",
    "International shipping",
]

df = df[~df["Location"].isin(aggregate_labels)]

df["Location"].head(20)


1                                  China
2                          United States
3                                  India
5                                 Russia
6                                  Japan
7                                   Iran
8                 International Shipping
9                              Indonesia
10                          Saudi Arabia
11                               Germany
12                                Canada
13                           South Korea
14                International Aviation
15                                Mexico
16                                Brazil
17                                Turkey
18                          South Africa
19                             Australia
20                               Vietnam
21    Italy, San Marino and Vatican City
Name: Location, dtype: object

In [7]:
# Step #6: Create a derived metric and show final table
# What it does: Uses the numeric columns to compute absolute emission change and prints a tidy readable dataset.

# Only compute if both columns exist
if all(c in df.columns for c in ["Emissions_2023_Mt", "Emissions_2000_Mt"]):
    df["Emissions_Change_Mt"] = df["Emissions_2023_Mt"] - df["Emissions_2000_Mt"]
    
# Clean tidy subset for display
keep_cols = ["Location"]
for c in ["Emissions_2023_Mt", "Emissions_2000_Mt", "Emissions_Change_Mt",
          "Percent_Global_Total", "Percent_Change_from_2000"]:
    if c in df.columns:
        keep_cols.append(c)

df_clean = df[keep_cols].sort_values(
    by=keep_cols[1], ascending=False
).reset_index(drop=True)

print("Clean CO₂ emissions dataset (first 20 rows):")
print(df_clean.head(20).to_string(index=False))

df_clean.info()


Clean CO₂ emissions dataset (first 20 rows):
                          Location  Emissions_2023_Mt  Emissions_2000_Mt  Emissions_Change_Mt  Percent_Global_Total  Percent_Change_from_2000
                             China           13259.64            3666.95              9592.69                  34.0                     262.0
                     United States            4682.04            5928.97             -1246.93                  12.0                       NaN
                             India            2955.18             995.65              1959.53                   7.6                     197.0
                            Russia            2069.50            1681.14               388.36                   5.3                      23.0
                             Japan             944.76            1248.81              -304.05                   2.4                       NaN
                              Iran             778.80             353.93               424.87          

In [9]:
df_clean.to_csv("cleaned_population.csv", index=False)


In [24]:
# Ethical Implications

'''
The data wrangling process included renaming columns for clarity, removing aggregate rows such as "World" to 
concentrate on country-level data, cleaning percentage fields by removing symbols and converting them to 
numeric types, and standardizing country names. These transformations were intended 
to enhance data usability and accuracy for analysis. The dataset is from Wikipedia, which gathers 
information from organizations like the Global Carbon Project. Although this data is public and 
does not have strict legal or constraints, it is important to avoid misinterpretation or 
misrepresentation, especially since emissions data can affect policy and public opinion. A risk 
from the cleaning steps is that errors—such as incorrect conversion of percentage signs or the removal of 
symbols could distort trends or misrepresent country emissions if not executed carefully. Assumptions made 
include trusting the accuracy of the Wikipedia source and believing that the removal of aggregate rows would 
not eliminate important context. To establish credibility, it is advisable to cross-reference the data with 
official reports. The data was ethically obtained through open-access sources, adhering to copyright and 
usage terms. To address potential ethical concerns, it is important to be transparent about the cleaning steps 
taken, and keep links to the original data. Lastly, it is important to communicate any assumptions and 
limitations in the cleaned data to prevent misleading conclusions.
'''


'\nThe data wrangling process included renaming columns for clarity, removing aggregate rows such as "World" to \nconcentrate on country-level data, cleaning percentage fields by removing symbols and converting them to \nnumeric types, and standardizing country names. These transformations were intended \nto enhance data usability and accuracy for analysis. The dataset is from Wikipedia, which gathers \ninformation from organizations like the Global Carbon Project. Although this data is public and \ndoes not have strict legal or constraints, it is important to avoid misinterpretation or \nmisrepresentation, especially since emissions data can affect policy and public opinion. A risk \nfrom the cleaning steps is that errors—such as incorrect conversion of percentage signs or the removal of \nsymbols could distort trends or misrepresent country emissions if not executed carefully. Assumptions made \ninclude trusting the accuracy of the Wikipedia source and believing that the removal of a